# 2. Train on 2×T4 (GPU notebook)

Settings: **Accelerator: GPU T4 ×2**, **Internet: On**. Add the
output of the data notebook as input (Add Input → Notebook). To **resume** a run, also add the
previous version of this notebook's output as input: the newest
`checkpoint.pt` found under `/kaggle/input` is copied in before training.

Long runs: **Save Version → Save & Run All (Commit)**, so the run continues
with the browser closed. Each config stops itself before the 12-hour limit.

In [ ]:
# --- Choose the run -------------------------------------------------------
CONFIG = "configs/pilot_124m.json"   # or configs/main_124m.json
RUN = "run_pilot"                    # folder under /kaggle/working
EXTRA = ""                           # e.g. "stop_after_steps=30 log_every=5 eval_every=30" for a smoke test

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 https://github.com/V1jey04/llm-from-scratch-124m.git /tmp/repo
!cd /tmp/repo && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "GPUs", torch.cuda.device_count())

In [ ]:
import glob, os, shutil
manifests = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert manifests, "Add the data notebook's output as an input."
DATA = os.path.dirname(manifests[0])
OUT = f"/kaggle/working/{RUN}"
os.makedirs(OUT, exist_ok=True)
previous = sorted(glob.glob(f"/kaggle/input/**/{RUN}/checkpoint.pt", recursive=True), key=os.path.getmtime)
if previous and not os.path.exists(f"{OUT}/checkpoint.pt"):
    source = os.path.dirname(previous[-1])
    for name in ("checkpoint.pt", "best_model.pt", "log.jsonl"):
        if os.path.exists(f"{source}/{name}"):
            shutil.copy(f"{source}/{name}", OUT)
    print("Resuming from", source)
print("data", DATA, "| out", OUT)

In [ ]:
!cd /tmp/repo && torchrun --standalone --nproc_per_node=2 -m lm.train --config {CONFIG} \
    --set data_dir={DATA} out_dir={OUT} {EXTRA}

In [ ]:
!ls -la {OUT} && tail -5 {OUT}/log.jsonl